<a href="https://colab.research.google.com/github/batinylmz/financialAnomalyDetection/blob/yunusMaster/TimeTransformerForFinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers

# ==========================================
# ADIM 1: VERİ HAZIRLIĞI VE CEVAP ANAHTARI
# ==========================================
df = pd.read_csv('/content/drive/MyDrive/financial_anomaly_benchmark_data.csv')

df['Anomaly_Score_Synthetic'] = np.abs(df['Volatility_HighLow'] * df['Volume_Change'])
threshold_true = np.percentile(df['Anomaly_Score_Synthetic'], 99)
df['True_Anomaly'] = (df['Anomaly_Score_Synthetic'] > threshold_true).astype(int)

df_sample = df.iloc[:20000].copy()

# ==========================================
# ADIM 2: KOPYASIZ ÖZELLİKLER
# ==========================================
df_sample['Intraday_Return'] = (df_sample['Close'] - df_sample['Open']) / df_sample['Open']
features = ['Returns', 'Volume', 'Intraday_Return']

X = df_sample[features].values
y = df_sample['True_Anomaly'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# ADIM 3: GELECEĞİ GÖRMEYEN (CAUSAL) ZAMAN PENCERELERİ
# ==========================================
SEQ_LEN = 10

def create_predictive_sequences(data, labels, seq_len):
    X_past, y_future_features, y_future_labels = [], [], []
    for i in range(len(data) - seq_len):
        # SADECE GEÇMİŞ (Örneğin t=0'dan t=9'a kadar)
        X_past.append(data[i:(i + seq_len)])

        # TAHMİN EDİLECEK GELECEK ADIM (t=10 anındaki gerçek değerler)
        y_future_features.append(data[i + seq_len])

        # O adımın gerçek anomali olup olmadığı (Test için lazım)
        y_future_labels.append(labels[i + seq_len])

    return np.array(X_past), np.array(y_future_features), np.array(y_future_labels)

X_seq, y_target_features, y_target_labels = create_predictive_sequences(X_scaled, y, SEQ_LEN)

# ==========================================
# ADIM 4: TAHMİNCİ (PREDICTIVE) TRANSFORMER MİMARİSİ
# ==========================================
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Dropout(dropout)(x)
    res = x + inputs

    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    return x + res

num_features = X_seq.shape[2]
inputs = tf.keras.Input(shape=(SEQ_LEN, num_features))

# 1. Geçmişi Transformer ile anla
x = transformer_encoder(inputs, head_size=32, num_heads=2, ff_dim=32, dropout=0.1)

# 2. Zaman dizisini tek bir özet vektöre sıkıştır (Geleceği tahmin etmek için)
x = layers.GlobalAveragePooling1D()(x)

# 3. GELECEK ADIMI TAHMİN ET (Çıktı boyutu artık seq_len değil, sadece num_features kadar)
outputs = layers.Dense(num_features)(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse")

# ==========================================
# ADIM 5: MODEL EĞİTİMİ (Sadece normal piyasa akışını öğrenir)
# ==========================================
print("\nGelecek Tahminli Transformer Eğitiliyor...")
# X_seq (Geçmiş) ile y_target_features (Gelecek Adım) eşleşiyor.
model.fit(X_seq, y_target_features, batch_size=64, epochs=5, validation_split=0.1, verbose=1)

# ==========================================
# ADIM 6: TAHMİN HATASI İLE ANOMALİ TESPİTİ
# ==========================================
# Modelden geçmişe bakarak gelecekte ne olacağını tahmin etmesini istiyoruz
y_pred_features = model.predict(X_seq)

# HATA = (Gerçekte Olan) - (Modelin Tahmin Ettiği)
# Eğer piyasa aniden çıldırırsa, modelin geçmişe dayalı sakin tahmini ile gerçek arasında devasa fark çıkar!
prediction_error = np.mean(np.square(y_target_features - y_pred_features), axis=1)

# Eşik belirleme (En çok yanıldığı %5'lik kısmı anomali say)
threshold_pred = np.percentile(prediction_error, 95)
y_pred_anomaly = (prediction_error > threshold_pred).astype(int)

# ==========================================
# ADIM 7: SONUÇLAR VE KIYASLAMA
# ==========================================
print("\n--- CAUSAL TRANSFORMER BAŞARISI (Eşik: %95) ---")
print("Hata Matrisi:")
print(confusion_matrix(y_target_labels, y_pred_anomaly))

print("\nDetaylı Sınıflandırma Raporu:")
print(classification_report(y_target_labels, y_pred_anomaly))


Gelecek Tahminli Transformer Eğitiliyor...
Epoch 1/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.7975 - val_loss: 1.9000
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.7761 - val_loss: 1.7823
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.7687 - val_loss: 1.7553
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 0.7663 - val_loss: 1.7254
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.7648 - val_loss: 1.7211
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

--- CAUSAL TRANSFORMER BAŞARISI (Eşik: %95) ---
Hata Matrisi:
[[18878   748]
 [  112   252]]

Detaylı Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98     19626
           1       0.25      0.69      0.37       364

    accuracy                           0.96     19990
   macro avg       0.62      0.83      0.67     19990
weighted avg       0.98      0.96      0.97     19990



Nedensel (Causal) Zaman Serisi Transformer ile Finansal Anomali Tespiti
Bu mimari, finansal zaman serilerindeki olağan dışı şokları (anomalileri) "veri sızıntısı" (geleceği görme hatası) olmadan tespit etmek için tasarlanmış derin öğrenme tabanlı bir tahmin (predictive) modelidir.

Nasıl Çalışır?

Geleceği Tahmin Etme (Causal Logic): Geleneksel modellerden farklı olarak bu mimari, veriye bütüncül bakmaz. Yalnızca geçmiş belirli bir zaman penceresini (örneğin ardışık 10 adet 15 dakikalık periyot) girdi olarak alır ve içindeki "Self-Attention" (Öz-Dikkat) mekanizmasıyla piyasa dinamiklerini öğrenerek bir sonraki adımı tahmin eder.

Anomali Tanımı: Modelin geçmişe bakarak oluşturduğu "sakin piyasa beklentisi" ile gerçekte yaşanan piyasa hareketi arasındaki fark (Hata/MSE) hesaplanır. Eğer bu hata payı belirlenen bir eşiğin (örneğin en yüksek %5'lik dilim) üzerindeyse, sistem bu durumu tahmin edilemeyen bir şok, yani Anomali olarak işaretler.

Modelin Temel Özellikleri ve Kullanımı

Kopyasız Eğitim: Model, test ve doğrulama için kullanılan sentetik etiket formülünü (Volatility * Volume_Change) eğitim sırasında kesinlikle görmez. Eğitim sadece Getiri (Returns), Hacim ve Gün İçi Getiri (Intraday Return) gibi bağımsız değişkenlerle yapılarak modelin "ezberlemesi" değil, piyasayı "öğrenmesi" sağlanır.

Nasıl Kullanılır: Kodu kendi verinizde çalıştırırken, modelin geçmişe ne kadar bakacağını SEQ_LEN (Pencere Boyutu) değişkeniyle ayarlayabilirsiniz. Modelin ne kadar agresif veya hassas anomali uyarısı vereceğini ise kodun sonundaki eşik değeri yüzdesini (örn: %95 veya %99) değiştirerek kontrol edebilirsiniz. Yüzde düştükçe (örn: 90) model daha fazla noktayı anomali olarak işaretleyecektir.